# Feature engineering - advanced data preparation pipeline  | Sebislaw

## Libraries

In [1]:
from os.path  import join
import random
import itertools
import math

import numpy as np
import pandas as pd
from pandas.plotting import scatter_matrix

import matplotlib.pyplot as plt
import plotly.express as px
from pandas.plotting import parallel_coordinates
import seaborn as sns
import ipywidgets as widgets
from IPython.display import display

from sklearn.linear_model import LinearRegression, LassoCV, LogisticRegression, LogisticRegressionCV
from sklearn.model_selection import train_test_split, TimeSeriesSplit, cross_val_score, StratifiedKFold, RandomizedSearchCV
import statsmodels.api as sm
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import brier_score_loss
from sklearn.feature_selection import SelectFromModel
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import mutual_info_classif
from sklearn.neural_network import MLPClassifier

import xgboost as xgb
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
import optuna
# from tabpfn import TabPFNClassifier

## Data

In [2]:
data_path = ''
pd.set_option('display.max_columns', None)

SampleSubmissionStage2 = pd.read_csv(join('..\\..\\..\\data', 'SampleSubmissionStage2.csv'))

MenTest = pd.read_csv(join(data_path, "MenTest.csv"), index_col=0)
MenTrain = pd.read_csv(join(data_path, "MenTrain.csv"), index_col=0)

WomenTest = pd.read_csv(join(data_path, "WomenTest.csv"), index_col=0)
WomenTrain = pd.read_csv(join(data_path, 'WomenTrain.csv'), index_col=0)

# MenTest and WomenTest have the same columns and the same
# 'Season', 'T1_TeamID', 'T1_Score', 'T2_TeamID', 'T2_Score', 'location'
# values in the same order, but Men have only daata for men teams and NaNs in other rows.
# The same for WomenTest.
# Test below fills the NaNs with values from the data frame, where values are present.
Test = MenTest.combine_first(WomenTest)

## Data preparation pipeline

In [3]:
def clear_na_from_x_y(x, y):
    """
    The data frame for final season is in format matching the submission file.
    This function clears NaNs from data.
    """
    # Create masks for training data:
    mask_train = ~np.isnan(x).any(axis=1) & ~np.isnan(y)
    x_clean = x[mask_train]
    y_clean = y[mask_train]
    return x_clean, y_clean

def x_y_from_data_frame(df):
    # Prepare data and labels
    x = df[list(df.columns[6:])].values
    y = np.where(
        df[['T1_Score', 'T2_Score']].isnull().any(axis=1),
        np.nan,
        np.where(df['T1_Score'] - df['T2_Score'] > 0, 1, 0)
    )
    return x, y

## Get data to use models on for testing

In [4]:
THE_LAST_YEAR = 2024
# I will be using train and test from train dataset
x_train_men, y_train_men = x_y_from_data_frame(MenTrain[MenTrain['Season']<THE_LAST_YEAR])
x_train_women, y_train_women = x_y_from_data_frame(WomenTrain[WomenTrain['Season']<THE_LAST_YEAR])
x_test_men, y_test_men = x_y_from_data_frame(MenTrain[MenTrain['Season']==THE_LAST_YEAR])
x_test_women, y_test_women = x_y_from_data_frame(WomenTrain[WomenTrain['Season']==THE_LAST_YEAR])

# Training models

In [5]:
import optuna
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import brier_score_loss
from sklearn.model_selection import StratifiedKFold

# Define an objective function that uses StratifiedKFold CV.
def objective_lr_cv(trial, X, y, n_folds=5):  # Increased folds for better generalization
    # Stronger regularization: Reduce range of C to prevent overfitting
    C = trial.suggest_float("C", 1e-4, 10, log=True)  # Lower upper bound
    
    penalty = trial.suggest_categorical("penalty", ["l1", "l2"])
    solver_all = "liblinear" if penalty == "l1" else "lbfgs"
    solver = solver_all  # More stable for both L1 and L2

    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=42)
    scores = []

    for train_idx, val_idx in skf.split(X, y):
        X_train_cv, X_val_cv = X[train_idx], X[val_idx]
        y_train_cv, y_val_cv = y[train_idx], y[val_idx]

        model = Pipeline([
            ("scaler", StandardScaler()),
            ("classifier", LogisticRegression(penalty=penalty, C=C, solver=solver, max_iter=3000))  # Increased max_iter
        ])

        model.fit(X_train_cv, y_train_cv)
        y_pred = model.predict_proba(X_val_cv)[:, 1]
        scores.append(brier_score_loss(y_val_cv, y_pred))

    return np.mean(scores)

###########################
# For all
###########################

x_train_all = np.concatenate((x_train_women, x_train_men))
y_train_all = np.concatenate((y_train_women, y_train_men))

x_test_all = np.concatenate((x_test_women, x_test_men))
y_test_all = np.concatenate((y_test_women, y_test_men))

###########################
study_all = optuna.create_study(direction="minimize")
study_all.optimize(lambda trial: objective_lr_cv(trial, x_train_all, y_train_all), n_trials=100)  # Increased trials for better tuning

best_params_all = study_all.best_params
print("Best Logistic Regression CV Params (all):", best_params_all)

best_lr_all = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(penalty=best_params_all["penalty"],
                                       C=best_params_all["C"],
                                       solver="saga",
                                       max_iter=3000))
])
best_lr_all.fit(x_train_all, y_train_all)
y_pred_all_lr = best_lr_all.predict_proba(x_test_all)[:, 1]
brier_all_lr = brier_score_loss(y_test_all, y_pred_all_lr)
print("Final Logistic Regression Brier Score (all):", brier_all_lr)

[I 2025-03-20 10:31:58,362] A new study created in memory with name: no-name-90610354-dff4-45b1-a734-9208e0834223
[I 2025-03-20 10:31:58,502] Trial 0 finished with value: 0.25 and parameters: {'C': 0.00016981944221209662, 'penalty': 'l1'}. Best is trial 0 with value: 0.25.
[I 2025-03-20 10:31:58,668] Trial 1 finished with value: 0.1505639411065847 and parameters: {'C': 0.05864774720064729, 'penalty': 'l1'}. Best is trial 1 with value: 0.1505639411065847.
[I 2025-03-20 10:31:59,231] Trial 2 finished with value: 0.14911964878997258 and parameters: {'C': 0.31107054133580786, 'penalty': 'l2'}. Best is trial 2 with value: 0.14911964878997258.
[I 2025-03-20 10:31:59,347] Trial 3 finished with value: 0.25 and parameters: {'C': 0.00011795097804350805, 'penalty': 'l1'}. Best is trial 2 with value: 0.14911964878997258.
[I 2025-03-20 10:31:59,575] Trial 4 finished with value: 0.1491174114630786 and parameters: {'C': 0.11919345120600237, 'penalty': 'l1'}. Best is trial 4 with value: 0.149117411463

[I 2025-03-20 10:32:22,109] Trial 44 finished with value: 0.14883124010030066 and parameters: {'C': 0.19434973235379419, 'penalty': 'l1'}. Best is trial 9 with value: 0.14879909158024107.
[I 2025-03-20 10:32:22,638] Trial 45 finished with value: 0.15062398594502197 and parameters: {'C': 0.05930937032497338, 'penalty': 'l2'}. Best is trial 9 with value: 0.14879909158024107.
[I 2025-03-20 10:32:23,791] Trial 46 finished with value: 0.1489215278608805 and parameters: {'C': 0.864571301734962, 'penalty': 'l1'}. Best is trial 9 with value: 0.14879909158024107.
[I 2025-03-20 10:32:24,495] Trial 47 finished with value: 0.1488474470709369 and parameters: {'C': 0.5554354427443888, 'penalty': 'l1'}. Best is trial 9 with value: 0.14879909158024107.
[I 2025-03-20 10:32:24,758] Trial 48 finished with value: 0.14922903685338415 and parameters: {'C': 0.1067909537479826, 'penalty': 'l1'}. Best is trial 9 with value: 0.14879909158024107.
[I 2025-03-20 10:32:25,380] Trial 49 finished with value: 0.149151

[I 2025-03-20 10:32:45,425] Trial 88 finished with value: 0.14933500337642933 and parameters: {'C': 0.09815937735908348, 'penalty': 'l1'}. Best is trial 62 with value: 0.14879660588488164.
[I 2025-03-20 10:32:45,898] Trial 89 finished with value: 0.14882740903970232 and parameters: {'C': 0.37511122542028386, 'penalty': 'l1'}. Best is trial 62 with value: 0.14879660588488164.
[I 2025-03-20 10:32:47,280] Trial 90 finished with value: 0.1491699393955222 and parameters: {'C': 6.150881780701622, 'penalty': 'l1'}. Best is trial 62 with value: 0.14879660588488164.
[I 2025-03-20 10:32:47,585] Trial 91 finished with value: 0.14879704323670565 and parameters: {'C': 0.2295424038664532, 'penalty': 'l1'}. Best is trial 62 with value: 0.14879660588488164.
[I 2025-03-20 10:32:47,851] Trial 92 finished with value: 0.14886130707833806 and parameters: {'C': 0.17601794313185412, 'penalty': 'l1'}. Best is trial 62 with value: 0.14879660588488164.
[I 2025-03-20 10:32:48,458] Trial 93 finished with value: 0

Best Logistic Regression CV Params (all): {'C': 0.234161426033768, 'penalty': 'l1'}
Final Logistic Regression Brier Score (all): 0.14740759773075035


## Testing the model

In [6]:
year_range = [i for i in range(2011, 2020)] + [i for i in range(2022, 2025)]
brier_all_list = []

for i in range(len(year_range)):
    
    THE_LAST_YEAR = year_range[i]
    # For data leak validation
#     print(len(MenTrain[MenTrain['Season']<THE_LAST_YEAR]['Season']))
#     print(len(MenTrain[MenTrain['Season']<THE_LAST_YEAR]['Season']))
#     print(len(MenTrain[MenTrain['Season']==THE_LAST_YEAR]))
#     print(MenTrain[MenTrain['Season']<THE_LAST_YEAR]['Season'].head())
#     print(MenTrain[MenTrain['Season']<THE_LAST_YEAR]['Season'].tail())
#     print(MenTrain[MenTrain['Season']==THE_LAST_YEAR]['Season'])
#     print("Predictions for season ", year_range[i])
    
    x_train_men, y_train_men = x_y_from_data_frame(MenTrain[MenTrain['Season']<THE_LAST_YEAR])
    x_train_women, y_train_women = x_y_from_data_frame(WomenTrain[WomenTrain['Season']<THE_LAST_YEAR])
    x_test_men, y_test_men = x_y_from_data_frame(MenTrain[MenTrain['Season']==THE_LAST_YEAR])
    x_test_women, y_test_women = x_y_from_data_frame(WomenTrain[WomenTrain['Season']==THE_LAST_YEAR])
    
    x_train_all = np.concatenate((x_train_women, x_train_men))
    y_train_all = np.concatenate((y_train_women, y_train_men))
    x_test_all = np.concatenate((x_test_women, x_test_men))
    y_test_all = np.concatenate((y_test_women, y_test_men))
    
    best_lr_all.fit(x_train_all, y_train_all)
    y_prob_all = best_lr_all.predict_proba(x_test_all)[:, 1]

    score = brier_score_loss(y_test_all, y_prob_all)
    brier_all_list.append(score)
    print('LogisticRegressionCV for All trained on All', score)
    # ----------------------------------------------
print()
print('The mean score when trained on All and tested on All was: ', np.mean(brier_all_list), 
      'with a std of ', np.std(brier_all_list))

# Best params {'C': 0.23059755790690561, 'penalty': 'l1'}

LogisticRegressionCV for All trained on All 0.15991023863814532
LogisticRegressionCV for All trained on All 0.13050991226085007
LogisticRegressionCV for All trained on All 0.1643632982993226
LogisticRegressionCV for All trained on All 0.14740839506320147
LogisticRegressionCV for All trained on All 0.12244743269136114
LogisticRegressionCV for All trained on All 0.15045226426678052
LogisticRegressionCV for All trained on All 0.13561557695165924
LogisticRegressionCV for All trained on All 0.1567095872355556
LogisticRegressionCV for All trained on All 0.1499856275370925
LogisticRegressionCV for All trained on All 0.14914271647213265
LogisticRegressionCV for All trained on All 0.16816014629771753
LogisticRegressionCV for All trained on All 0.14740906867567113

The mean score when trained on All and tested on All was:  0.14850952203245749 with a std of  0.012951693856833748


# Create Submission

In [28]:
data_path = ''
pd.set_option('display.max_columns', None)

SampleSubmissionStage2 = pd.read_csv(join('..\\..\\..\\data', 'SampleSubmissionStage2.csv'))

MenTest = pd.read_csv(join(data_path, "MenTest.csv"), index_col=0)
MenTrain = pd.read_csv(join(data_path, "MenTrain.csv"), index_col=0)

WomenTest = pd.read_csv(join(data_path, "WomenTest.csv"), index_col=0)
WomenTrain = pd.read_csv(join(data_path, 'WomenTrain.csv'), index_col=0)

# MenTest and WomenTest have the same columns and the same
# 'Season', 'T1_TeamID', 'T1_Score', 'T2_TeamID', 'T2_Score', 'location'
# values in the same order, but Men have only daata for men teams and NaNs in other rows.
# The same for WomenTest.
# Test below fills the NaNs with values from the data frame, where values are present.
Test = MenTest.combine_first(WomenTest)

In [29]:
x_test, y_test = x_y_from_data_frame(Test)

In [30]:
# I will fill the NaNs (seeds) with 0. It doesn't matter, since those teams won't play in the tournament anyway.
x_test = np.nan_to_num(x_test, nan=0)

In [31]:
predictions = best_lr_all.predict_proba(x_test)[:, 1]

In [32]:
Submission = SampleSubmissionStage2.copy()

In [33]:
Submission['Pred'] = predictions

In [34]:
Submission

,ID,Pred
0,2025_1101_1102,0.808613
1,2025_1101_1103,0.105368
2,2025_1101_1104,0.004942
3,2025_1101_1105,0.953795
4,2025_1101_1106,0.925149
...,...,...
131402,2025_3477_3479,0.289967
131403,2025_3477_3480,0.128249
131404,2025_3478_3479,0.539758
131405,2025_3478_3480,0.297001


In [35]:
Submission.to_csv("LogisticRegressionWithElo.csv", index=False)

In [38]:
pd.read_csv(join(data_path, "LogisticRegressionWithElo.csv"))

,ID,Pred
0,2025_1101_1102,0.808613
1,2025_1101_1103,0.105368
2,2025_1101_1104,0.004942
3,2025_1101_1105,0.953795
4,2025_1101_1106,0.925149
...,...,...
131402,2025_3477_3479,0.289967
131403,2025_3477_3480,0.128249
131404,2025_3478_3479,0.539758
131405,2025_3478_3480,0.297001
